Trying to figure out which callback function works

In [4]:
# Initialise a SpeedyWeather simulation
using SpeedyWeather, CairoMakie, GLMakie
spectral_grid = SpectralGrid()
model = PrimitiveWetModel(spectral_grid)
simulation = initialize!(model)

# Add radiation and surface flux
add!(model, SpeedyWeather.RadiationOutput()...)
add!(model, SpeedyWeather.SurfaceFluxesOutput()...)

# Run the simulation for one year and plot the sensible heat flux
run!(simulation, period=Day(365))
heatmap(simulation.diagnostic_variables.physics.sensible_heat_flux)

In [13]:
# Helpful functions

# Default long names for Trenberth variables
const TRENBERTH_LONGNAMES = Dict(
    :LHF => "Surface latent heat flux (W/m²)",
    :SHF => "Surface sensible heat flux (W/m²)",
    :SSRU => "Surface shortwave up (W/m²)",
    :SLRU => "Surface longwave up (W/m²)",
    :SSRD => "Surface shortwave down (W/m²)",
    :SLRD => "Surface longwave down (W/m²)",
    :OSR => "Outgoing shortwave radiation (TOA) (W/m²)",
    :OLR => "Outgoing longwave radiation (TOA) (W/m²)",
    :albedo => "Surface albedo",
    :SW_net_sfc => "Surface net shortwave (W/m²)",
    :LW_net_sfc => "Surface net longwave (W/m²)",
    :surface_net => "Surface net energy (W/m²)"
)

# Pretty-print the long names
function show_var_names(cb::TrenberthCallback)
    for (k, long) in cb.var_longnames
        println(string(k), " → ", long)
    end
    return nothing
end

# Optional: assemble a DataFrame if DataFrames.jl is installed
function to_dataframe(cb::TrenberthCallback)
    try
        @eval using DataFrames
    catch
        error("DataFrames.jl not available. Install it with `using Pkg; Pkg.add(\"DataFrames\")`")
    end
    df = DataFrame(time = cb.datetimes)
    for (k, vec) in cb.data
        colname = get(cb.var_longnames, k, string(k))  # column name uses long name if available
        # ensure column identifier is a Symbol
        df[Symbol(colname)] = vec
    end
    return df
end

using Dates

# Convert various time types to Float64. Default unit = :seconds.
function time_to_float(t; unit::Symbol = :seconds)
    if t isa DateTime
        secs = Dates.datetime2unix(t)                     # seconds since Unix epoch
        return unit == :seconds ? Float64(secs) :
               unit == :days    ? Float64(secs / 86400.0) :
               error("unsupported unit: $unit")
    elseif t <: Dates.Period   # Day, Hour, Minute, etc.
        # Dates.value returns the integer magnitude in the Period's base units
        # For Day it returns number of days, for Hour number of hours, etc.
        # Convert to days or seconds depending on unit
        if unit == :days
            return float(Dates.value(t))
        elseif unit == :seconds
            # approximate: convert days/hours etc. to seconds using common ratios
            # We'll convert via Day/Hr/Minute explicitly for safety:
            if t isa Day
                return float(Dates.value(t) * 86400)
            elseif t isa Hour
                return float(Dates.value(t) * 3600)
            elseif t isa Minute
                return float(Dates.value(t) * 60)
            else
                # fallback: convert to days then seconds
                return float(Dates.value(Day(round(Int, Dates.value(t)))) * 86400)
            end
        else
            error("unsupported unit: $unit")
        end
    elseif t isa Number
        return float(t)
    else
        error("unsupported time type: $(typeof(t))")
    end
end


time_to_float (generic function with 1 method)

In [14]:
# Create a calculator function which computes Trenberth fluxes from diagn + model

function calc_trenberth_from_diagn(diagn, model; SumFlag::Bool=false)
    fields = Dict(
        :LHF   => diagn.physics.surface_latent_heat_flux,
        :SHF   => diagn.physics.sensible_heat_flux,
        :SSRU  => diagn.physics.surface_shortwave_up,
        :SLRU  => diagn.physics.surface_longwave_up,
        :SSRD  => diagn.physics.surface_shortwave_down,
        :SLRD  => diagn.physics.surface_longwave_down,
        :OSR   => diagn.physics.outgoing_shortwave_radiation,
        :OLR   => diagn.physics.outgoing_longwave_radiation,
        :albedo=> diagn.physics.albedo
    )

    # spectral helpers (ℓ=0 → global mean; multiply by area for total)
    function calc_global_mean(field)
        a = transform(field)              # model transform -> spectral coeffs
        a00 = real(a[1])                  # index 1 == ℓ=0,m=0 (SpeedyWeather layout)
        return a00 / model.spectral_transform.norm_sphere
    end
    function calc_global_sum(field)
        mean_val = calc_global_mean(field)
        area = 4π * model.planet.radius^2
        return mean_val * area
    end

    calcfun = SumFlag ? calc_global_sum : calc_global_mean

    results = Dict{Symbol, Float64}()
    for (k, f) in fields
        try
            results[k] = Float64(calcfun(f))
        catch err
            @warn "calc_trenberth_from_diagn: could not compute $k: $err"
            results[k] = NaN
        end
    end

    # derived surface/Trenberth terms (adjust sign convention as needed)
    results[:SW_net_sfc]  = results[:SSRD] - results[:SSRU]
    results[:LW_net_sfc]  = results[:SLRD] - results[:SLRU]
    results[:surface_net] = results[:SW_net_sfc] + results[:LW_net_sfc] - results[:LHF] - results[:SHF]

    return results
end

calc_trenberth_from_diagn (generic function with 1 method)

In [ ]:
# Define the TrenberthCallback function to store data

function TrenberthCallback(; vars = [:LHF,:SHF,:SSRU,:SLRU,:SSRD,:SLRD,:OSR,:OLR,:albedo,:SW_net_sfc,:LW_net_sfc,:surface_net],
                             SumFlag::Bool=false,
                             nsteps::Int=0,
                             var_longnames::Dict{Symbol,String}=TRENBERTH_LONGNAMES)
    d = Dict{Symbol, Vector{Float64}}()
    for v in vars
        d[v] = nsteps > 0 ? Vector{Float64}(undef, nsteps + 1) : Float64[]
    end
    times = nsteps > 0 ? Vector{Float64}(undef, nsteps + 1) : Float64[]
    datetimes = nsteps > 0 ? Vector{DateTime}(undef, nsteps + 1) : DateTime[]
    return TrenberthCallback(0, d, times, datetimes, 0.0, SumFlag, var_longnames)
end

TrenberthCallback

In [36]:
# Run the callback for the initialise step

function SpeedyWeather.initialize!(cb::TrenberthCallback,
                                   progn::PrognosticVariables,
                                   diagn::DiagnosticVariables,
                                   model::AbstractModel)
    # Store the simulation start time for reference (convert DateTime to Float64 Unix timestamp)
    cb.start_time = Dates.datetime2unix(progn.clock.time)
    
    # Try to get nsteps, but if it doesn't work, just start with empty vectors
    try
        nsteps = progn.clock.nsteps
        # if our data dict vectors are empty or wrong size, (re)allocate
        for (k, v) in cb.data
            if isempty(v) || length(v) != nsteps + 1
                cb.data[k] = Vector{Float64}(undef, nsteps + 1)
            end
        end
        if isempty(cb.times) || length(cb.times) != nsteps + 1
            cb.times = Vector{Float64}(undef, nsteps + 1)
        end
        if isempty(cb.datetimes) || length(cb.datetimes) != nsteps + 1
            cb.datetimes = Vector{DateTime}(undef, nsteps + 1)
        end
    catch
        # If we can't get nsteps, just use dynamic push mode
        @info "Could not determine nsteps, using dynamic push mode"
    end

    # set counter to 1 and store initial conditions
    cb.timestep_counter = 1
    t0 = Dates.datetime2unix(progn.clock.time)  # Convert DateTime to Unix timestamp
    dt0 = progn.clock.time  # Get the original DateTime object
    # compute initial values using diagn
    res0 = calc_trenberth_from_diagn(diagn, model; SumFlag=cb.SumFlag)
    for (k, v) in res0
        if haskey(cb.data, k)
            if length(cb.data[k]) > 0
                cb.data[k][1] = v
            else
                push!(cb.data[k], v)
            end
        else
            cb.data[k] = [v]
        end
    end

    # Store time relative to simulation start (in seconds) and DateTime
    if length(cb.times) > 0
        cb.times[1] = t0 - cb.start_time
        cb.datetimes[1] = dt0
    else
        push!(cb.times, t0 - cb.start_time)
        push!(cb.datetimes, dt0)
    end
    return nothing
end

In [37]:
# Run callback after every timestep (once completed)

function SpeedyWeather.callback!(cb::TrenberthCallback,
                                 progn::PrognosticVariables,
                                 diagn::DiagnosticVariables,
                                 model::AbstractModel)
    # increment step index
    cb.timestep_counter += 1
    
    # compute current diagnostics
    res = calc_trenberth_from_diagn(diagn, model; SumFlag=cb.SumFlag)
    
    # push new values to the arrays
    for (k, v) in res
        if !haskey(cb.data, k)
            # new key appeared: create vector and push
            cb.data[k] = [v]
        else
            # existing key: push to the vector
            push!(cb.data[k], v)
        end
    end
    
    # record model time relative to simulation start (in seconds) and DateTime
    # Convert DateTime to Unix timestamp, then subtract start_time to get elapsed seconds
    current_time = Dates.datetime2unix(progn.clock.time)
    push!(cb.times, current_time - cb.start_time)
    push!(cb.datetimes, progn.clock.time)  # Store the DateTime object
    return nothing
end

In [38]:
# Finalise simulation

SpeedyWeather.finalize!(cb::TrenberthCallback, args...) = nothing

In [50]:
# Add callback to the model

# Dynamic push mode (flexible):
#cb = TrenberthCallback(SumFlag=false,  nsteps=0)

# preallocated (fast) if you know nsteps:
cb = TrenberthCallback(SumFlag=false, nsteps=0)

# A: add into the callbacks dict directly (works if model.callbacks is a Dict-like)
add!(model.callbacks, :trenberth => cb)

keys(model.callbacks)              # should include :trenberth
model.callbacks[:trenberth] === cb # should be true

sim = initialize!(model)   # this will call SpeedyWeather.initialize! on cb
run!(sim, period=Day(10))  # or your usual run invocation

cb.data

┌ Info: Could not determine nsteps, using dynamic push mode
└ @ Main c:\Users\Hannah\OneDrive - Nexus365\Documents\group_project\speedyweather_trenberth_diagram\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_W5sZmlsZQ==.jl:27


Dict{Symbol, Vector{Float64}} with 12 entries:
  :LW_net_sfc  => [0.0, -239.742, -238.96, -238.311, -237.911, -237.739, -237.8…
  :OLR         => [0.0, 357.659, 352.521, 350.784, 346.932, 343.66, 342.421, 34…
  :SSRD        => [0.0, 341.258, 341.252, 341.259, 341.256, 341.267, 341.243, 3…
  :SLRD        => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.…
  :SHF         => [0.0, 20.9445, 25.3991, 22.7005, 19.4452, 19.1563, 15.0698, 1…
  :SSRU        => [0.0, 28.0432, 28.3989, 29.1781, 30.0129, 30.8464, 31.6498, 3…
  :SLRU        => [0.0, 239.742, 238.96, 238.311, 237.911, 237.739, 237.803, 23…
  :LHF         => [0.0, 14.0758, 14.1463, 15.1494, 14.0292, 12.547, 9.97612, 7.…
  :albedo      => [0.0, 0.115173, 0.115173, 0.115173, 0.115173, 0.115173, 0.115…
  :SW_net_sfc  => [0.0, 313.214, 312.853, 312.081, 311.243, 310.421, 309.593, 3…
  :OSR         => [0.0, 28.0432, 28.3989, 29.1781, 30.0129, 30.8464, 31.6498, 3…
  :surface_net => [0.0, 38.4521, 34.3478, 35.9207, 39.8579, 40

In [ ]:
# Plotting data to check for bugs 
using CairoMakie, GLMakie
plot(cb.data[:OLR])